# 07 — Data Export (CSV, Excel, JSON, Parquet, SQL)

Maps to **MODULE-07a / 07b**.

Docs: `datasets/README.md` (Phase 7)

In [ ]:
import pandas as pd
from pathlib import Path

Path('reports').mkdir(exist_ok=True)

orders = pd.read_csv('datasets/raw/orders.csv', parse_dates=['order_date'])

### 1. Build a summary to export

In [ ]:
monthly = orders.set_index('order_date').resample('ME').agg(
    revenue=('total', 'sum'),
    orders=('order_id', 'count'))
monthly.head()

### 2. CSV

In [ ]:
monthly.to_csv('reports/monthly_sales.csv', index_label='month')
pd.read_csv('reports/monthly_sales.csv').head()

### 3. Excel — multi-sheet report

In [ ]:
by_status = orders.groupby('status')['total'].agg(['count', 'sum'])
with pd.ExcelWriter('reports/sales_report.xlsx', engine='openpyxl') as writer:
    monthly.to_excel(writer, sheet_name='Monthly')
    by_status.to_excel(writer, sheet_name='By Status')
print('wrote reports/sales_report.xlsx')

### 4. JSON

In [ ]:
by_status.to_json('reports/by_status.json', orient='records', indent=2)
print('wrote reports/by_status.json')

### 5. Parquet — a fast checkpoint

In [ ]:
orders.to_parquet('reports/orders_clean.parquet')
pd.read_parquet('reports/orders_clean.parquet').dtypes   # dtypes preserved

### 6. SQLite

In [ ]:
from sqlalchemy import create_engine
engine = create_engine('sqlite:///reports/techretail.db')
orders.to_sql('orders', engine, index=False, if_exists='replace')
pd.read_sql('SELECT status, COUNT(*) AS n FROM orders GROUP BY status', engine)